# Multiclass classification: newswires

Forty-six classes, and the information bottleneck that appears when an intermediate layer is smaller than the output.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 4 — Classification and Regression](../../../course-web-slides/ch04/index.html) &nbsp;·&nbsp; **Section:** 02 — Classifying newswires

---

## The data

In [ ]:
from keras.datasets import reuters
import numpy as np

(train_data, train_labels), (test_data, test_labels) = reuters.load_data(
    num_words=10000
)
print(len(train_data), "training,", len(test_data), "test")
print("classes:", len(set(train_labels)))
print("class distribution (top 5):",
      np.bincount(train_labels).argsort()[::-1][:5])

Expected output:

```
8982 training, 2246 test
classes: 46
class distribution (top 5): [ 3  4 19 16  1]
```

## Vectorizing inputs and targets

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension), dtype="float32")
    for i, seq in enumerate(sequences):
        for j in seq:
            results[i, j] = 1.
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

# Two equally valid ways to encode the targets.
y_train_int = np.asarray(train_labels)
y_test_int = np.asarray(test_labels)

import keras
y_train_oh = keras.utils.to_categorical(train_labels)
print("integer targets:", y_train_int.shape, " one-hot:", y_train_oh.shape)

> **Note** — One-hot targets go with `categorical_crossentropy`; integer targets go with `sparse_categorical_crossentropy`. **Same loss, different interface** — pairing them wrongly produces a shape error, which is the good outcome.

## The model

In [ ]:
from keras import layers

model = keras.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(46, activation="softmax"),
])
model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

x_val, partial_x = x_train[:1000], x_train[1000:]
y_val, partial_y = y_train_int[:1000], y_train_int[1000:]

history = model.fit(partial_x, partial_y, epochs=20, batch_size=512,
                    validation_data=(x_val, y_val), verbose=2)

Sixty-four units, not sixteen. With 46 output classes, a **16-unit layer would be an information bottleneck** — the next cell demonstrates that rather than asserting it.

## The bottleneck, demonstrated

In [ ]:
import matplotlib.pyplot as plt

def run(units, epochs=20):
    keras.utils.set_random_seed(0)
    m = keras.Sequential([
        layers.Dense(units, activation="relu"),
        layers.Dense(units, activation="relu"),
        layers.Dense(46, activation="softmax"),
    ])
    m.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    h = m.fit(partial_x, partial_y, epochs=epochs, batch_size=512,
              validation_data=(x_val, y_val), verbose=0)
    return h.history["val_accuracy"]

plt.figure(figsize=(7, 4.4))
for u in [4, 16, 64]:
    plt.plot(run(u), lw=1.5, label=f"{u} hidden units")
plt.xlabel("epoch"); plt.ylabel("validation accuracy")
plt.legend(); plt.title("An intermediate layer smaller than the output loses information")
plt.show()

Four units cannot carry 46 classes' worth of separation, and no amount of training recovers it. **Information dropped by a layer is never recovered by a later one** — the layers form a pipeline, not a committee.

## A baseline worth computing

In [ ]:
import copy
test_labels_copy = copy.copy(test_labels)
np.random.shuffle(test_labels_copy)
random_baseline = float((np.array(test_labels) == np.array(test_labels_copy)).mean())
majority = np.bincount(train_labels).max() / len(train_labels)

model.fit(x_train, y_train_int, epochs=9, batch_size=512, verbose=0)
_, acc = model.evaluate(x_test, y_test_int, verbose=0)

print(f"random guessing:  {random_baseline:.3f}")
print(f"always the most common class: {majority:.3f}")
print(f"this model:       {acc:.3f}")

Expected output:

```
random guessing:  0.0xx
always the most common class: 0.36x
this model:       0.79x
```

**Beating random is not the bar.** The majority-class baseline is 36% here, and it costs one line to compute. Chapter 6 makes this a required step of the workflow.

---

## What to take away

- N-way classification: N units, softmax, and a crossentropy loss.
- One-hot targets pair with `categorical_crossentropy`; integers with the `sparse_` variant.
- **An intermediate layer smaller than the output is a permanent bottleneck.**
- Compute the majority-class baseline before believing any accuracy figure.